## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

from src.ingestion.csv_loader import CSVLoader
from src.utils.config import Config

print("✅ Imports successful")

## 2. Load Configuration

In [ ]:
config = Config()
print(f"Data directory: {config.data_dir}")
print(f"Sample data: {config.data_dir}/sample/sample_data.csv")

## 3. Load CSV Data

We'll use our CSVLoader class to load test results.

In [ ]:
# Initialize loader
loader = CSVLoader()

# Load sample data
sample_path = config.data_dir / 'sample' / 'sample_data.csv'
df = loader.load(sample_path)

print(f"\n✅ Loaded {len(df):,} records")
print(f"   Devices: {df['device_id'].nunique()}")
print(f"   Tests: {df['test_name'].nunique()}")
df.head()

## 4. Inspect Data Structure

In [ ]:
# Check data types
print("Data Types:")
print(df.dtypes)

print("\nMemory Usage:")
print(f"{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Statistical summary
df.describe()

## 5. Data Quality Checks

In [ ]:
# Check for missing values
print("Missing Values:")
missing = df.isnull().sum()
print(missing[missing > 0])

if missing.sum() == 0:
    print("✅ No missing values found")

In [ ]:
# Check for duplicates
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

if duplicates == 0:
    print("✅ No duplicates found")

In [ ]:
# Check data consistency
print("Data Consistency Checks:")
print(f"Unique lots: {df['lot_id'].nunique()}")
print(f"Unique wafers: {df['wafer_id'].nunique()}")
print(f"Unique devices: {df['device_id'].nunique()}")
print(f"Tests per device: {len(df) / df['device_id'].nunique():.0f}")

## 6. Visualize Data Distribution

In [ ]:
# Pass/Fail distribution
fig = px.histogram(
    df, 
    x='result', 
    title='Test Result Distribution',
    color='result',
    color_discrete_map={'pass': 'green', 'fail': 'red'}
)
fig.show()

In [ ]:
# Test time distribution
fig = px.histogram(
    df, 
    x='test_time_ms',
    nbins=30,
    title='Test Time Distribution',
    labels={'test_time_ms': 'Test Time (ms)'}
)
fig.show()

In [ ]:
# Test yield by test name
test_yield = df.groupby('test_name')['result'].apply(
    lambda x: (x == 'pass').sum() / len(x) * 100
).sort_values()

fig = px.bar(
    x=test_yield.values,
    y=test_yield.index,
    orientation='h',
    title='Yield by Test',
    labels={'x': 'Yield (%)', 'y': 'Test Name'}
)
fig.show()

## 7. Save Processed Data

In [ ]:
# Save to staging area (parquet format)
staging_dir = config.data_dir / 'staging'
staging_dir.mkdir(exist_ok=True)

output_path = staging_dir / 'test_results.parquet'
df.to_parquet(output_path, index=False)

print(f"✅ Saved processed data to {output_path}")
print(f"   File size: {output_path.stat().st_size / 1024:.1f} KB")

## 8. Summary

**What we learned:**
- ✅ Loading CSV data with validation
- ✅ Inspecting data structure and types
- ✅ Checking data quality (missing values, duplicates)
- ✅ Visualizing data distributions
- ✅ Saving data in efficient format (Parquet)

**Next Steps:**
- Notebook 02: Advanced data quality and profiling
- Implement automated data validation rules
- Handle different data formats (STDF binary)